In [1]:
import pandas as pd

raw = pd.read_csv("webtoon_raw.csv")
detail = pd.read_csv("webtoon_detail.csv")
print("raw:", raw.shape, "/ detail:", detail.shape)

# titleId를 열쇠로 두 표를 옆으로 붙이기
df = pd.merge(raw, detail, on="titleId", how="inner")
print("병합 결과:", df.shape)
df.head(3)

raw: (4103, 20) / detail: (3872, 6)
병합 결과: (3926, 25)


,titleId,titleName,author,writers,painters,novelOriginAuthors,thumbnailUrl,up,rest,bm,...,bestChallengeLevelUp,finish,new,week,status,synopsis,favoriteCount,tags,firstDate,totalCount
0,758037,참교육,채용택 / 한가람,"[{'id': 301243, 'name': '채용택', 'blogUrl': 'htt...","[{'id': 314484, 'name': '한가람', 'blogUrl': ''}]",[],https://image-comic.pstatic.net/webtoon/758037...,False,False,False,...,False,False,False,mon,연재중,무너진 교권을 지키기 위해 교권보호국 소속 나화진의 참교육이 시작된다!\n<부활남>...,1061667,"액션, 사이다, 학원액션, 자극적인, 사회고발, 참교육, 블루스트링, 학원물, 다크...",20.11.01,257
1,822657,환생천마,JP / 부겸 / 장영훈,"[{'id': 348256, 'name': 'JP'}]","[{'id': 323121, 'name': '부겸'}]","[{'id': 376586, 'name': '장영훈'}]",https://image-comic.pstatic.net/webtoon/822657...,False,False,False,...,False,False,False,mon,연재중,"철혈의 맹주, 강호의 절대자 '천하진'. 가문의 수치라 불리는 망나니 '벽리단'의 ...",263765,"무협/사극, 사이다, 동양, 환생, 액션, 고인물, 환골탈태, 동양풍판타지, 판무,...",24.03.17,131
2,840014,샤MONEY즘,나락 / 영기,"[{'id': 319300, 'name': '나락'}]","[{'id': 358227, 'name': '영기'}]",[],https://image-comic.pstatic.net/webtoon/840014...,False,False,False,...,False,False,False,mon,연재중,“진짜 일류 무당은 여의도에서 논다.” \n주가의 오르내림을 예측하는 것만으로 수십...,376199,"판타지, 오컬트판타지, 자본주의, 인생역전",25.06.29,59


In [2]:
# titleId별 등장 횟수 세기 → 2번 이상인 것들
dup_counts = df["titleId"].value_counts()
print("중복된 작품 수:", (dup_counts >= 2).sum())

# 중복 작품 하나를 골라 실제로 확인
dup_ids = dup_counts[dup_counts >= 2].index
df[df["titleId"] == dup_ids[0]][["titleId", "titleName", "week", "status"]]

중복된 작품 수: 25


,titleId,titleName,week,status
466,842647,용사파티만화,fri,연재중
467,842647,용사파티만화,fri,연재중
468,842647,용사파티만화,fri,연재중
579,842647,용사파티만화,sat,연재중
580,842647,용사파티만화,sat,연재중
581,842647,용사파티만화,sat,연재중
704,842647,용사파티만화,sun,연재중
705,842647,용사파티만화,sun,연재중
706,842647,용사파티만화,sun,연재중


In [3]:
# titleId 기준 중복 제거 (첫 번째 행만 남김)
df = df.drop_duplicates(subset="titleId", keep="first")
print("중복 제거 후:", df.shape)

중복 제거 후: (3846, 25)


In [4]:
# firstDate에서 연도 추출: "20.11.01" → 2020
df["year"] = 2000 + df["firstDate"].str.split(".").str[0].astype(int)

# 연도별 작품 수 확인
print(df["year"].value_counts().sort_index())

year
2005      2
2006      7
2007     18
2008     37
2009     46
2010     60
2011     67
2012     65
2013     80
2014     81
2015     89
2016     83
2017     79
2018    121
2019    187
2020    254
2021    383
2022    438
2023    420
2024    490
2025    529
2026    310
Name: count, dtype: int64


In [5]:
# 먼치킨 라벨: 태그 또는 소개글에 '먼치킨' 포함 여부
df["is_munchkin"] = (
    df["tags"].str.contains("먼치킨", na=False) |
    df["synopsis"].str.contains("먼치킨", na=False)
)

print("먼치킨 작품:", df["is_munchkin"].sum(), "개")

# 태그로만 잡은 것과 비교 (소개글 보조 기준이 몇 개를 더 잡았나)
tag_only = df["tags"].str.contains("먼치킨", na=False).sum()
print("태그 기준:", tag_only, "개 → 소개글 추가로", df["is_munchkin"].sum() - tag_only, "개 더 잡음")

먼치킨 작품: 282 개
태그 기준: 281 개 → 소개글 추가로 1 개 더 잡음


In [6]:
df.to_csv("webtoon_clean.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", df.shape)

저장 완료: (3846, 27)
